In [1]:
import sys

sys.path.append("../src")

from recommendation import generate_recommendation

In [2]:
#Test a HIGH irrigation case
result = generate_recommendation(
    irrigation_need="High",
    soil_moisture=20,
    rain_forecast=10,
    temperature=35,
    humidity=42,
    rainfall=1
)

print(result)

{'status': 'Irrigation Required', 'priority': 'High', 'action': 'Irrigate now', 'irrigation_need': 'High', 'reason': 'AI model detected high irrigation requirement. Soil moisture is critically low. Temperature is high. Humidity is relatively low.', 'soil_moisture': 20.0, 'rain_forecast': 10.0}


In [3]:
#Test MEDIUM
result = generate_recommendation(
    irrigation_need="Medium",
    soil_moisture=32,
    rain_forecast=15,
    temperature=30,
    humidity=60,
    rainfall=3
)

print(result)

{'status': 'Irrigation Recommended', 'priority': 'Medium', 'action': 'Plan irrigation soon', 'irrigation_need': 'Medium', 'reason': 'AI model detected moderate irrigation requirement. Significant rainfall is not expected.', 'soil_moisture': 32.0, 'rain_forecast': 15.0}


In [4]:
#Test LOW
result = generate_recommendation(
    irrigation_need="Low",
    soil_moisture=45,
    rain_forecast=10,
    temperature=27,
    humidity=70,
    rainfall=5
)

print(result)

{'status': 'No Immediate Irrigation', 'priority': 'Low', 'action': 'Wait and monitor', 'irrigation_need': 'Low', 'reason': 'AI model detected low irrigation requirement under the current conditions.', 'soil_moisture': 45.0, 'rain_forecast': 10.0}


In [5]:
#Test heavy rain
result = generate_recommendation(
    irrigation_need="High",
    soil_moisture=20,
    rain_forecast=85,
    temperature=30,
    humidity=65,
    rainfall=2
)

print(result)

{'status': 'Delay Irrigation', 'priority': 'Low', 'action': 'Do not irrigate now', 'irrigation_need': 'High', 'reason': 'High rainfall is expected.', 'soil_moisture': 20.0, 'rain_forecast': 85.0}


In [ ]:
                USER INPUT
                     │
                     ↓
               XGBOOST MODEL
                     │
                     ↓
             Low / Medium / High
                     │
                     ↓
          ┌─────────────────────┐
          │ Recommendation      │
          │ Engine              │
          └──────────┬──────────┘
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
 Soil Moisture   Rain Forecast  Weather
        │            │            │
        └────────────┼────────────┘
                     ↓
               FINAL DECISION
                     │
        ┌────────────┼──────────────┐
        ↓            ↓              ↓
      Status       Priority       Reason 

In [6]:
#Connect ML prediction + recommendation engine

import joblib
import pandas as pd
import sys

sys.path.append("../src")

from recommendation import generate_recommendation

In [9]:
import os
import joblib

model_path = "../models/irrigation_need_model.pkl"

print("Model exists:", os.path.exists(model_path))

model = joblib.load(model_path)

print("Model loaded successfully!")

Model exists: True
Model loaded successfully!


In [10]:
Low = 0
Medium = 1
High = 2

In [11]:
label_mapping = {
    0: "Low",
    1: "Medium",
    2: "High"
}

print(label_mapping)

{0: 'Low', 1: 'Medium', 2: 'High'}


In [12]:
model_package = {
    "model": model,
    "label_mapping": label_mapping
}

package_path = "../models/irrigation_model_package.pkl"

joblib.dump(
    model_package,
    package_path
)

print("Model package saved successfully!")
print("Location:", package_path)

Model package saved successfully!
Location: ../models/irrigation_model_package.pkl


In [13]:
package = joblib.load(
    "../models/irrigation_model_package.pkl"
)

model = package["model"]
label_mapping = package["label_mapping"]

print("Package loaded successfully!")
print("Label mapping:", label_mapping)

Package loaded successfully!
Label mapping: {0: 'Low', 1: 'Medium', 2: 'High'}


In [15]:
import pandas as pd   #Test the model

df = pd.read_csv(
    "../data/irrigation_prediction.csv"
)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

display(df.head())

,soil_type,soil_ph,soil_moisture,organic_carbon,electrical_conductivity,temperature_c,humidity,rainfall_mm,sunlight_hours,wind_speed_kmh,crop_type,crop_growth_stage,season,irrigation_type,water_source,field_area_hectare,mulching_used,previous_irrigation_mm,region,irrigation_need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


In [16]:
X = df.drop(
    columns=[
        "irrigation_need",
        "irrigation_type",
        "water_source"
    ],
    errors="ignore"
)

print(X.columns.tolist())  #Create the same features used for training

['soil_type', 'soil_ph', 'soil_moisture', 'organic_carbon', 'electrical_conductivity', 'temperature_c', 'humidity', 'rainfall_mm', 'sunlight_hours', 'wind_speed_kmh', 'crop_type', 'crop_growth_stage', 'season', 'field_area_hectare', 'mulching_used', 'previous_irrigation_mm', 'region']


In [18]:
sample = X.iloc[[0]]

print(sample)  #take one row

  soil_type  soil_ph  soil_moisture  organic_carbon  electrical_conductivity  \
0      Clay     6.14          36.48            0.42                     2.17   

   temperature_c  humidity  rainfall_mm  sunlight_hours  wind_speed_kmh  \
0           21.9     31.19       1167.7            4.01            1.97   

  crop_type crop_growth_stage season  field_area_hectare mulching_used  \
0     Wheat        Vegetative   Rabi                4.73           Yes   

   previous_irrigation_mm region  
0                    1.98  South  


In [19]:
prediction = model.predict(sample)

print("Encoded prediction:", prediction)

Encoded prediction: [0]


In [20]:
irrigation_need = label_mapping[
    int(prediction[0])
]

print("Irrigation Need:", irrigation_need)

Irrigation Need: Low


In [22]:
import sys  #connect the Recommendation Engine

sys.path.append("../src")

from recommendation import generate_recommendation

In [23]:
soil_moisture = float(                         #the required values:
    sample["soil_moisture"].iloc[0]
)

temperature = float(
    sample["temperature_c"].iloc[0]
)

humidity = float(
    sample["humidity"].iloc[0]
)

rainfall = float(
    sample["rainfall_mm"].iloc[0]
)

In [24]:
rain_forecast = 10  #For now, use a sample rain forecast

In [25]:
recommendation = generate_recommendation(
    irrigation_need=irrigation_need,
    soil_moisture=soil_moisture,
    rain_forecast=rain_forecast,
    temperature=temperature,
    humidity=humidity,
    rainfall=rainfall
)

print(recommendation)

{'status': 'No Immediate Irrigation', 'priority': 'Low', 'action': 'Wait and monitor', 'irrigation_need': 'Low', 'reason': 'AI model detected low irrigation requirement under the current conditions.', 'soil_moisture': 36.48, 'rain_forecast': 10.0}


In [ ]:
Your pipeline is now working
Dataset
   ↓
Saved XGBoost Model
   ↓
Prediction
   ↓
Low / Medium / High
   ↓
Recommendation Engine
   ↓
Status + Priority + Action + Reason